# Exploratory Data Analysis for the fraud detection POC.
## Covers: missing values, class imbalance, dtypes, cleaning, and a stratified train/test split.
 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
import os
from pathlib import Path

# Change working directory to project root
project_root = Path(r"c:\Users\dell\OneDrive\Desktop\Fraud-Detection-System")
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'c:\\Users\\dell\\OneDrive\\Desktop\\Fraud-Detection-System'

In [7]:
# ---------------------------------------------------------------------------
# 1. Load the data
# ---------------------------------------------------------------------------
DATA_PATH = "data/raw/creditcard.csv"
 
df = pd.read_csv(DATA_PATH)
 
print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

SHAPE
Rows: 284,807  |  Columns: 31


In [9]:
# ---------------------------------------------------------------------------
# 2. Data types
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
print(df.dtypes.value_counts())
print()
print(df.dtypes)
 
# What to look for:
# - V1-V28 and Amount/Time should all be float64/int64. If any column shows up
#   as 'object', it likely has stray strings, mixed types, or got corrupted
#   during export -- inspect that column with df['col'].unique() before doing
#   anything else.
# - 'Class' should be int64 (0/1). If it's float, cast it: df['Class'].astype(int)


DATA TYPES
float64    30
int64       1
Name: count, dtype: int64

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object


In [11]:
# ---------------------------------------------------------------------------
# 3. Missing values
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0].sort_values(
    "missing_count", ascending=False
)
 
if missing_report.empty:
    print("No missing values found.")
else:
    print(missing_report)
 
# What to look for:
# - The ULB Credit Card Fraud dataset is normally complete (0 missing values).
#   If you see missingness here, it's likely from a partial download, a bad
#   unzip, or you're using a different/modified copy of the data -- re-download
#   before proceeding rather than imputing blindly.
# - If you switch to IEEE-CIS later, expect heavy missingness (some columns
#   >90% null) -- that dataset needs an explicit imputation strategy.


MISSING VALUES
No missing values found.


In [14]:
# ---------------------------------------------------------------------------
# 4. Class imbalance
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("CLASS BALANCE (Class: 0 = legit, 1 = fraud)")
print("=" * 60)
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100
 
print(f"Legitimate: {class_counts[0]:,} ({class_pct[0]:.4f}%)")
print(f"Fraud:      {class_counts[1]:,} ({class_pct[1]:.4f}%)")
print(f"Imbalance ratio (legitimate:fraud) -> "f"{class_counts[0] / class_counts[1]:.0f}:1")
 
# What to look for:
# - Expect roughly 99.83% legit vs 0.17% fraud (~578:1 ratio) on the ULB dataset.
# - This confirms your LLD's note on class imbalance (SMOTE / class weights) is
#   not optional -- a model that just predicts "legit" every time would score
#   99.8% accuracy while being useless. This is exactly why your LLD says to
#   use precision/recall/F1/AUC-ROC instead of accuracy.
# - Do NOT balance the classes before splitting into train/test. Balance
#   (SMOTE, class_weight, undersampling) only on the TRAINING set, after the
#   split -- otherwise you leak synthetic/duplicated fraud examples into your
#   test set and get falsely inflated evaluation metrics.


CLASS BALANCE (Class: 0 = legit, 1 = fraud)
Legitimate: 284,315 (99.8273%)
Fraud:      492 (0.1727%)
Imbalance ratio (legitimate:fraud) -> 578:1


In [16]:
# ---------------------------------------------------------------------------
# 5. Duplicate rows
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("DUPLICATES")
print("=" * 60)
dupe_count = df.duplicated().sum()
print(f"Fully duplicated rows: {dupe_count}")
 
# What to look for:
# - The raw ULB CSV has ~1,081 exact duplicate rows. These are almost
#   certainly data artifacts (not real repeat transactions with 28 identical
#   PCA components), and they inflate whichever split they land in more
#   heavily. Drop them before splitting.



DUPLICATES
Fully duplicated rows: 1081


In [17]:
# ---------------------------------------------------------------------------
# 6. Basic distribution checks (Amount, Time)
# ---------------------------------------------------------------------------
print("\n" + "=" * 60)
print("AMOUNT / TIME SUMMARY")
print("=" * 60)
print(df[["Time", "Amount"]].describe())
 
# What to look for:
# - Amount is heavily right-skewed (a few very large transactions, many small
#   ones) -- most models benefit from a log1p transform on Amount at feature
#   engineering time (in features.py), not here.
# - Time is seconds elapsed since the first transaction, spanning ~2 days.
#   Negative or absurdly large values would indicate a corrupted export.
 


AMOUNT / TIME SUMMARY
                Time         Amount
count  284807.000000  284807.000000
mean    94813.859575      88.349619
std     47488.145955     250.120109
min         0.000000       0.000000
25%     54201.500000       5.600000
50%     84692.000000      22.000000
75%    139320.500000      77.165000
max    172792.000000   25691.160000


In [18]:
# ---------------------------------------------------------------------------
# 7. Cleaning
# ---------------------------------------------------------------------------
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"\nRemoved {len(df) - len(df_clean)} duplicate rows.")
print(f"Clean shape: {df_clean.shape}")
 
# Only clean what's actually broken -- don't invent work. On this dataset
# that's just deduping. If your missing-value check above found anything,
# handle it explicitly per-column rather than a blanket dropna()/fillna().


Removed 1081 duplicate rows.
Clean shape: (283726, 31)


In [20]:
# ---------------------------------------------------------------------------
# 8. Train/test split (stratified -- critical given the imbalance)
# ---------------------------------------------------------------------------
X = df_clean.drop(columns=["Class"])
y = df_clean["Class"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # <- preserves the fraud ratio in both splits
)
 
print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)
print(f"Train: {X_train.shape[0]:,} rows | Fraud: {y_train.sum()} ({y_train.mean()*100:.4f}%)")
print(f"Test:  {X_test.shape[0]:,} rows | Fraud: {y_test.sum()} ({y_test.mean()*100:.4f}%)")
 
# What to look for:
# - stratify=y is not optional here. Without it, a random split can easily
#   put a disproportionate share of the ~490 fraud cases into one split,
#   skewing both training and evaluation.
# - Confirm both printed fraud percentages are close to the original ~0.17%.
#   If they're wildly different, something's wrong with the split (e.g. you
#   split before dedup, or forgot stratify).


TRAIN / TEST SPLIT
Train: 226,980 rows | Fraud: 378 (0.1665%)
Test:  56,746 rows | Fraud: 95 (0.1674%)


In [21]:
# ---------------------------------------------------------------------------
# 9. Save the splits for the next step (train.py)
# ---------------------------------------------------------------------------
X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)
 
print("\nSaved splits to data/processed/")


Saved splits to data/processed/
